# OCR via OpenAI gpt-4o-mini vision

Fast, accurate OCR for the small-claims PDFs using OpenAI's vision endpoint instead of a local model.

## Why this exists

Local Qwen2-VL on RunPod produced token-loop garbage; local MLX on Mac would take ~20 hours. OpenAI gpt-4o-mini vision does the same job in **~30-45 minutes for ~$4-5 total**, with quality matching Qwen2-VL-7B for SC-100 forms.

## How to run

1. **Activate the env**: `conda activate ML`
2. **Make sure Drive Desktop is running** and the `litigation-pipeline` folder is set to Available Offline so PDFs are local.
3. **Open this notebook**, run cells 1 → 6 top to bottom.
4. **Cell 3** will prompt for your OpenAI API key (or set `OPENAI_API_KEY` env var first).
5. **Cell 5** is the long one — kicks off OCR for all PDFs missing a `.txt`. Watch the progress prints. Drive Desktop auto-syncs new `.txt` files to Google Drive within ~1 min.

## Resume / re-run

Cell 2 skips any PDF that already has a `.txt` companion in `extracted/`, so killing the kernel mid-run and re-running is safe — just re-run from Cell 2.

In [1]:
from pathlib import Path

DRIVE_DIR = Path(
    "/Users/ernesto/Library/CloudStorage/"
    "GoogleDrive-ernestod1998@gmail.com/My Drive/litigation-pipeline"
)
PDF_DIR = DRIVE_DIR / "pdfs"
OUTPUT_DIR = DRIVE_DIR / "extracted"

# Set to small int (e.g. 5) for a test run before committing to the full batch.
LIMIT = None

# Whitelist mirrors scraper/config.py::DOC_TYPE_WHITELIST.
DOC_TYPE_WHITELIST = {
    "CLAIM_OF_PLAINTIFF",
    "DEFENDANT_S_CLAIM",
    "JUDGMENT",
    "ORDER",
    "DISMISSAL",
    "Notice_of_Entry_of_Judgment",
    "DECLARATION_OF_APPEARANCE",
    "STIPULATION",
    "COURT_JUDGMENT",
}


def is_doc_type_wanted(description: str) -> bool:
    desc_upper = description.upper()
    return any(w.upper() in desc_upper for w in DOC_TYPE_WHITELIST)


def _doc_type(p):
    return p.stem.split("_", 1)[-1] if "_" in p.stem else p.stem


assert PDF_DIR.exists(), f"PDF_DIR not found: {PDF_DIR}. Is Drive Desktop running and folder set Available Offline?"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_pdfs = sorted(PDF_DIR.glob("*.pdf"))
whitelisted = [p for p in all_pdfs if is_doc_type_wanted(_doc_type(p))]
needs_ocr = [p for p in whitelisted if not (OUTPUT_DIR / f"{p.stem}.txt").exists()]

if LIMIT:
    needs_ocr = needs_ocr[:LIMIT]

print(f"PDFs total:                {len(all_pdfs)}")
print(f"Whitelisted doc types:     {len(whitelisted)}")
print(f"Already extracted to .txt: {len(whitelisted) - len(needs_ocr) if not LIMIT else '(filtered)'}")
print(f"To OCR this run:           {len(needs_ocr)}{' (LIMIT=' + str(LIMIT) + ')' if LIMIT else ''}")

PDFs total:                8732
Whitelisted doc types:     8617
Already extracted to .txt: 7220
To OCR this run:           1397


In [2]:
import os
import getpass
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (input hidden): ").strip()
assert api_key, "No API key provided"

client = OpenAI(api_key=api_key)

# Quick sanity ping — list models, fail fast if the key is bad.
_ = client.models.list()
print("OpenAI client ready.")

OpenAI client ready.


In [3]:
import base64
import io

import fitz

MODEL = "gpt-4o-mini"
DPI = 150
DETAIL = "high"  # "high" = better OCR on dense forms (~1500-2000 tokens/page); "low" = ~85 tokens/page but loses detail
MAX_TOKENS = 2048

GENERIC_PROMPT = (
    "This is a page from a San Francisco small claims court document. "
    "Extract all text exactly as it appears. Include party names, dates, "
    "claim amounts, rulings, and any other information on the page. "
    "Output plain text only."
)


def page_to_png_b64(page, dpi=DPI):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return base64.b64encode(pix.tobytes("png")).decode("ascii")


def ocr_page(b64_image):
    """Send one image to gpt-4o-mini, return text + usage dict.
    OpenAI SDK retries transient errors 3x automatically."""
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": GENERIC_PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{b64_image}",
                            "detail": DETAIL,
                        },
                    },
                ],
            }
        ],
    )
    return response.choices[0].message.content, {
        "input": response.usage.prompt_tokens,
        "output": response.usage.completion_tokens,
    }


def ocr_pdf(pdf_path):
    """OCR all pages of a PDF. Returns (text, usage, error_or_None)."""
    try:
        doc = fitz.open(str(pdf_path))
    except Exception as e:
        return None, {"input": 0, "output": 0}, f"open failed: {e}"

    pages_text = []
    total_in, total_out = 0, 0
    try:
        for page in doc:
            b64 = page_to_png_b64(page)
            text, usage = ocr_page(b64)
            pages_text.append(text)
            total_in += usage["input"]
            total_out += usage["output"]
    except Exception as e:
        doc.close()
        return None, {"input": total_in, "output": total_out}, f"OCR failed: {e}"

    doc.close()
    return (
        "\n\n--- PAGE BREAK ---\n\n".join(pages_text),
        {"input": total_in, "output": total_out},
        None,
    )


print(f"Configured: model={MODEL}, detail={DETAIL}, dpi={DPI}, max_tokens={MAX_TOKENS}")

Configured: model=gpt-4o-mini, detail=high, dpi=150, max_tokens=2048


In [ ]:
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_CONCURRENT = 1  # PDFs in flight; well under tier-1 RPM limits at ~3-5s per page

# gpt-4o-mini pricing (per 1M tokens) — January 2026 published rates
PRICE_IN_PER_1M = 0.150
PRICE_OUT_PER_1M = 0.600


def process(pdf_path):
    txt_path = OUTPUT_DIR / f"{pdf_path.stem}.txt"
    if txt_path.exists():
        return pdf_path, "exists", {"input": 0, "output": 0}, None
    text, usage, err = ocr_pdf(pdf_path)
    if err:
        return pdf_path, "error", usage, err
    txt_path.write_text(text, encoding="utf-8")
    return pdf_path, "ok", usage, None


_lock = threading.Lock()
totals = {"input": 0, "output": 0, "ok": 0, "errors": 0}


def fmt_cost(in_tok, out_tok):
    return f"${in_tok / 1e6 * PRICE_IN_PER_1M + out_tok / 1e6 * PRICE_OUT_PER_1M:.2f}"


t0 = time.time()
errors = []
n = len(needs_ocr)

with ThreadPoolExecutor(max_workers=MAX_CONCURRENT) as ex:
    futures = {ex.submit(process, p): p for p in needs_ocr}
    for i, fut in enumerate(as_completed(futures), start=1):
        pdf_path, status, usage, err = fut.result()
        with _lock:
            totals["input"] += usage["input"]
            totals["output"] += usage["output"]

        if status == "ok":
            totals["ok"] += 1
            txt_path = OUTPUT_DIR / f"{pdf_path.stem}.txt"
            kb = txt_path.stat().st_size / 1024
            running_cost = fmt_cost(totals["input"], totals["output"])
            print(f"[{i}/{n}] {pdf_path.name} ({kb:.1f} KB) — total so far: {running_cost}")
        elif status == "error":
            totals["errors"] += 1
            errors.append((pdf_path.name, err))
            print(f"[{i}/{n}] FAIL {pdf_path.name}: {err}")
        # status == "exists" → silent skip (shouldn't happen since needs_ocr filtered them out, but just in case)

elapsed = time.time() - t0
final_cost = fmt_cost(totals["input"], totals["output"])
print(f"\n{'='*60}")
print(f"Done in {elapsed/60:.1f} min")
print(f"  OK:     {totals['ok']}")
print(f"  Errors: {totals['errors']}")
print(f"  Input tokens:  {totals['input']:,}")
print(f"  Output tokens: {totals['output']:,}")
print(f"  Total cost:    {final_cost}")
if errors:
    print(f"\nFirst 10 errors:")
    for name, err in errors[:10]:
        print(f"  {name}: {err}")

[1/1397] CSM24867795_CLAIM_OF_PLAINTIFF.pdf (12.5 KB) — total so far: $0.02
[2/1397] CSM24867796_CLAIM_OF_PLAINTIFF.pdf (7.5 KB) — total so far: $0.05
[3/1397] CSM24867797_CLAIM_OF_PLAINTIFF.pdf (9.8 KB) — total so far: $0.07
[4/1397] CSM24867799_CLAIM_OF_PLAINTIFF.pdf (8.3 KB) — total so far: $0.10
[5/1397] CSM24867800_CLAIM_OF_PLAINTIFF.pdf (8.5 KB) — total so far: $0.13
[6/1397] CSM24867802_CLAIM_OF_PLAINTIFF.pdf (6.5 KB) — total so far: $0.15
[7/1397] CSM24867803_CLAIM_OF_PLAINTIFF.pdf (7.5 KB) — total so far: $0.17
[8/1397] FAIL CSM24867804_CLAIM_OF_PLAINTIFF.pdf: OCR failed: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-f1KAS0AFLT1zkPGqmAhrJ7z6 on tokens per min (TPM): Limit 200000, Used 200000, Requested 822. Please try again in 246ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
[9/1397] FAIL CSM24867805_CLAIM_OF_PLAINTIFF.pdf: OCR failed:

In [ ]:
# Summary + spot-check the first new .txt to make sure it's coherent.
all_txts = sorted(OUTPUT_DIR.glob("*.txt"))
total_bytes = sum(f.stat().st_size for f in all_txts)

print(f"Total .txt files in extracted/: {len(all_txts):,}")
print(f"Total size:                     {total_bytes / 1024 / 1024:.1f} MB")
print()
print("Drive Desktop auto-syncs new files to Google Drive within ~1 min.")
print(f"Folder: {OUTPUT_DIR}")
print()

# Spot-check a recently-written CSM24 claim file (verify real text, not garbage)
sample_candidates = sorted(
    OUTPUT_DIR.glob("CSM24*_CLAIM_OF_PLAINTIFF.txt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if sample_candidates:
    sample = sample_candidates[0]
    text = sample.read_text(encoding="utf-8")
    print(f"Spot-check: {sample.name} ({len(text):,} chars)")
    print("-" * 60)
    print(text[:800])
    print("-" * 60)
    print("Look for: plaintiff/defendant names, addresses, dollar amounts, dates.")
    print("NOT: 'mlmlml', '[src[src', or any repeated token pattern.")